<a href="https://colab.research.google.com/github/pulipulichen/Colab-PDF-Protector/blob/main/Colab_PDF_Protector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Colab-PDF-Protector

In [ ]:
# @markdown 如果是帳單，開啟密碼通常是身份證字號。<br />
# @markdown 如果是高鐵，通常是乘車日期YYYYMMDD。
OPEN_PASSWORD = "open_password" # @param {"type":"string"}
EDIT_PASSWORD = "edit_password" # @param {"type":"string"}
PDF_ACTION = "Decrypt" # @param ["Encrypt","Decrypt"]

# @markdown 是否使用「檔名分隔符號前的內容」作為開啟密碼（預設開啟）
USE_FILENAME_PREFIX_AS_OPEN_PASSWORD = True # @param {"type":"boolean"}
# @markdown 檔名分隔符號（例如：帳號_202607.pdf 的分隔符號是 _）
FILENAME_PASSWORD_DELIMITER = "_" # @param {"type":"string"}

# 多個markdown下拉式設true or false，true的話，就會自動刪除已經處理成功的pdf
DELETE_SUCCESS_PDF = True # @param {"type":"boolean"}




# Process

In [ ]:
import os
import subprocess


def resolve_open_password(file_name):
    if not USE_FILENAME_PREFIX_AS_OPEN_PASSWORD:
        return OPEN_PASSWORD

    file_stem = os.path.splitext(file_name)[0]
    if FILENAME_PASSWORD_DELIMITER:
        password = file_stem.split(FILENAME_PASSWORD_DELIMITER, 1)[0]
    else:
        password = file_stem

    return password if password else OPEN_PASSWORD


def main():
    # Step 1: Install QPDF
    !dpkg -s qpdf >/dev/null 2>&1 || (apt-get -y update && apt-get -y install qpdf)

    # Step 2: Create an output directory
    output_dir = "/content/output"
    os.makedirs(output_dir, exist_ok=True)

    for file_name in os.listdir(output_dir):
        file_path = os.path.join(output_dir, file_name)
        if os.path.isfile(file_path):
            os.remove(file_path)

    # Step 3: Unlock PDFs in the /content/ directory
    input_dir = "/content/"
    got_error = False
    success_input_paths = []

    for file_name in os.listdir(input_dir):
        if file_name.endswith(".pdf"):
            input_path = os.path.join(input_dir, file_name)
            output_path = os.path.join(output_dir, f"{file_name[:-4]}_unlock.pdf")
            open_password = resolve_open_password(file_name)

            # Run QPDF command to unlock/encrypt PDF
            if PDF_ACTION == "Decrypt":
                command = [
                    "qpdf",
                    "--decrypt",
                    f"--password={open_password}",
                    input_path,
                    output_path,
                ]
            else:
                output_path = os.path.join(output_dir, f"{file_name[:-4]}_lock.pdf")
                command = [
                    "qpdf",
                    "--encrypt",
                    open_password,
                    EDIT_PASSWORD,
                    "256",
                    "-extract=n",
                    "-modify=none",
                    "-annotate=n",
                    "--",
                    input_path,
                    output_path,
                ]

            subprocess.run(command, check=False)

            if os.path.exists(output_path):
                print(f"Processed: {file_name} -> {output_path}")
                success_input_paths.append(input_path)
            else:
                print(f"Error in processing: {file_name}")
                got_error = True

    if DELETE_SUCCESS_PDF:
        for file_path in success_input_paths:
            if os.path.isfile(file_path):
                os.remove(file_path)

    if got_error is False:
        print("All PDFs have been processed. Check the /content/output directory.")
    else:
        print("Some PDFs got errors. Please check console message.")





# Main

In [ ]:

main()